In [ ]:

# Luego importamos otras librerias que seguramente vamos a necesitar

import pandas as pd
import numpy as np
import re
import unicodedata
#from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
df_train = pd.read_csv("train.csv")
data = df_train.copy()


(12000, 3)

## 1. Procesamiento de texto

In [ ]:
stop_words_es = set(stopwords.words('spanish'))

# Palabras relevantes que no queremos eliminar porque cambiarian la interpretación del modelo
palabras_relevantes = {"no", "nunca", "jamás", "sin", "ni", "tampoco",
"pero", "aunque","muy", "bastante", "demasiado", "más", "menos", "poco"}

stop_words_es -= palabras_relevantes

stemmer = SnowballStemmer('spanish')

def preprocesar_texto(texto):
    # A. Normalización: Convertir a minúsculas
    texto = texto.lower()
    
    # B. Normalización: Quitar signos de puntuación (deja solo letras y espacios)
    texto = re.sub(r'[^\w\s]', '', texto)
    
    # C. Tokenización: Separar por espacios
    tokens = texto.split()
    
    # D. Stopwords y Stemming: Filtrar palabras vacías y extraer la raíz
    tokens_procesados = [
        stemmer.stem(palabra) 
        for palabra in tokens 
        if palabra not in stop_words_es
    ]
    
    # E. Volver a unir en una sola cadena para el vectorizador
    return " ".join(tokens_procesados)

# 3. Aplicar el pipeline al DataFrame
df_train['text_limpio'] = df_train['text'].apply(preprocesar_texto)

# 4. Vectorizar (Bolsa de Palabras)
vec = CountVectorizer(
    max_features=10000, 
    min_df=3
)

X_train = vec.fit_transform(df_train['text_limpio'])

In [ ]:


# 1. Normalización de texto
def normalizar_texto(texto):
    texto = str(texto)
    texto = unicodedata.normalize("NFKC", texto)
    texto = texto.lower()
    texto = re.sub(r"[^\w\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


df["text_normalized"] = df["text"].fillna("").apply(normalizar_texto)

# Visualización de cambios
display(df[["text", "text_normalized"]].head(3))

,text,text_normalized
0,Lo pedí un martes por la noche y me llegó al d...,lo pedí un martes por la noche y me llegó al d...
1,Se lo compré a un familiar después de pensarla...,se lo compré a un familiar después de pensarla...
2,Lo pedí a mediados de mes y me llegó en tres d...,lo pedí a mediados de mes y me llegó en tres d...


In [ ]:
# 2. Tokenización
df["tokens"] = df["text_normalized"].str.split()

display(df[["text_normalized", "tokens"]].head(3))

,text_normalized,tokens
0,lo pedí un martes por la noche y me llegó al d...,"[lo, pedí, un, martes, por, la, noche, y, me, ..."
1,se lo compré a un familiar después de pensarla...,"[se, lo, compré, a, un, familiar, después, de,..."
2,lo pedí a mediados de mes y me llegó en tres d...,"[lo, pedí, a, mediados, de, mes, y, me, llegó,..."


In [9]:
# 3. Stopwords: eliminar ruido

stop_words = {
    "a", "al", "con", "como", "de", "del", "desde", "durante",
    "el", "en", "entre", "la", "las", "lo", "los", "para", "por",
    "que", "se", "su", "sus", "un", "una", "unas", "uno", "unos",
    "y"
}

# Palabras relevantes para sentimiento que no queremos eliminar, si hay más de una
palabras_relevantes = {
    "aunque", "bastante", "demasiado", "jamás", "más", "menos", "ni",
    "no", "nunca", "pero", "poco", "sin", "tampoco", "muy"
}

stop_words -= palabras_relevantes

df["tokens_sin_stopwords"] = df["tokens"].apply(
    lambda tokens: [token for token in tokens if token not in stop_words]
)

display(df[["tokens", "tokens_sin_stopwords"]].head(3))

,tokens,tokens_sin_stopwords
0,"[lo, pedí, un, martes, por, la, noche, y, me, ...","[pedí, martes, noche, me, llegó, día, siguient..."
1,"[se, lo, compré, a, un, familiar, después, de,...","[compré, familiar, después, pensarla, varias, ..."
2,"[lo, pedí, a, mediados, de, mes, y, me, llegó,...","[pedí, mediados, mes, me, llegó, tres, días, c..."
